Migrated from Databricks to Neuron (auto-port).

In [ ]:
import os
import sys
from pathlib import Path

for _candidate in (
    "/home/jovyan/work/default/rashmi-ranjan/lib",
    str(Path.cwd().parent / "lib"),
    str(Path.cwd() / "lib"),
):
    if _candidate not in sys.path and Path(_candidate).is_dir():
        sys.path.insert(0, _candidate)

from neuron_runtime import (
    GoogleSheetReader,
    SlackReportsTrigger,
    get_sql_df,
    load_service_account,
    load_slack_token,
    make_unique_columns,
)

service_account = load_service_account()
slack_token = load_slack_token()

In [ ]:
import pandas as pd

query = """
WITH base AS (
    SELECT
        CAST(substr(t.ticket_creation_time, 1, 10) AS DATE) AS ticket_date,
        t.canonical_ticket_id
    FROM corp_analytics.cx_master_data_model_tbl t
    WHERE t.product_group = 'BBPS'
      AND t.source_channel = 'CHAT'
      AND t.overall_last_agent_type_for_ticket = 'GEN_AI_VOICE_BOT'
)

SELECT
    ticket_date AS Date,
    COUNT(*) AS "Total Calls"
FROM base
WHERE ticket_date >= current_date - INTERVAL '60' DAY
  AND ticket_date <= current_date - INTERVAL '1' DAY
GROUP BY ticket_date
ORDER BY ticket_date DESC
"""

result = get_sql_df(query)

if isinstance(result, tuple):
    df, code = result
else:
    df = result
    code = None

if df is not None and not df.empty:
    display(df)
else:
    print("No rows returned.", f"Code={code}" if code is not None else "")

In [ ]:
# Python cell
import os
import shutil

if not os.path.exists("data"):
    os.makedirs("data")

gsheet_url = "https://docs.google.com/spreadsheets/d/1TgIrX7t3inQ6iWeMbLM8ENGdWUsRs1L-fyrE0EskF1U/edit?gid=857594292#gid=857594292"
worksheet_name = "Connected Call Count"
gs_reader = GoogleSheetReader(gsheet_url, service_account)

gs_reader.clear_sheets(
    worksheet_name,
    start_row=1,
    start_column=1
)

combined_df = pd.concat([df], ignore_index=True) 

gs_reader.set_df_in_sheets( 
worksheet_name, 
combined_df, 
include_headers=True, 
row=1, 
col=1
)